# TDA-PiToMe vs PiToMe: Performance Benchmark

This notebook compares **TDA-PiToMe** (Topological Data Analysis-based token merging) with the original **PiToMe** on image classification tasks.

**Metrics:**
- Accuracy (Top-1)
- GFLOPs (computational efficiency)
- Throughput (images/sec)

## 1. Setup

In [ ]:
# Clone repository
!rm -rf PiToMe
!git clone -b feature/tda-pitome https://github.com/a11to1n3/PiToMe.git
%cd PiToMe

In [ ]:
# Install dependencies
!pip install -q timm accelerate wandb datasets torchvision pillow scikit-image

In [ ]:
import torch
import time
import numpy as np
from timm import create_model
from timm.data import resolve_data_config, create_transform
from PIL import Image
import requests
from io import BytesIO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Model and Apply Patches

In [ ]:
# Import patch modules
from algo import pitome, tda_pitome

# Model configuration
MODEL_NAME = 'deit_small_patch16_224'  # Options: deit_tiny, deit_small, deit_base
RATIO = 0.9  # Token keep ratio (0.9 = merge 10% of tokens)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Model: {MODEL_NAME}")
print(f"Ratio: {RATIO}")
print(f"Device: {DEVICE}")

In [ ]:
def create_patched_model(model_name, algo, ratio, device):
    """Create a model with the specified token merging algorithm."""
    model = create_model(model_name, pretrained=True)
    
    if algo == 'pitome':
        pitome.patch.deit(model)
    elif algo == 'tda_pitome':
        tda_pitome.patch.deit(model)
    else:
        pass  # No patching (baseline)
    
    model.ratio = ratio
    model = model.to(device)
    model.eval()
    return model

# Create transform
baseline_model = create_model(MODEL_NAME, pretrained=True)
config = resolve_data_config({}, model=baseline_model)
transform = create_transform(**config)
del baseline_model

## 3. Download Sample Images

In [ ]:
# Sample ImageNet images (or use your own)
SAMPLE_URLS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/b/bc/Juvenile_Ragdoll.jpg/1200px-Juvenile_Ragdoll.jpg",
]

def load_image(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    return transform(img)

# Create a batch of images
images = torch.stack([load_image(url) for url in SAMPLE_URLS])
print(f"Loaded {len(images)} images, shape: {images.shape}")

## 4. Benchmark Functions

In [ ]:
@torch.no_grad()
def benchmark_model(model, images, device, n_runs=50, warmup=10):
    """
    Benchmark a model's throughput and get FLOPs.
    
    Returns:
        throughput: images per second
        avg_flops: average FLOPs per image
    """
    images = images.to(device)
    
    # Warmup
    for _ in range(warmup):
        _ = model(images)
    
    # Synchronize before timing
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    total_flops = 0
    start = time.perf_counter()
    
    for _ in range(n_runs):
        output = model(images)
        
        # Get FLOPs if available
        if isinstance(output, tuple) and len(output) == 2:
            logits, flops = output
            total_flops += flops
        elif hasattr(model, 'total_flop'):
            total_flops += model.total_flop
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    
    total_images = n_runs * len(images)
    throughput = total_images / elapsed
    avg_flops = total_flops / n_runs / len(images) if total_flops > 0 else 0
    
    return throughput, avg_flops

In [ ]:
@torch.no_grad()
def get_predictions(model, images, device):
    """Get model predictions for visualization."""
    images = images.to(device)
    output = model(images)
    
    if isinstance(output, tuple):
        logits = output[0]
    else:
        logits = output
    
    probs = torch.softmax(logits, dim=-1)
    top5_probs, top5_indices = probs.topk(5, dim=-1)
    
    return top5_probs.cpu(), top5_indices.cpu()

## 5. Run Benchmark

In [ ]:
# Benchmark configurations
RATIOS = [0.9, 0.925, 0.95, 0.975]
ALGOS = ['none', 'pitome', 'tda_pitome']

results = []

for algo in ALGOS:
    for ratio in RATIOS:
        if algo == 'none' and ratio != 1.0:
            # Baseline only runs once
            if ratio != RATIOS[0]:
                continue
            ratio = 1.0
        
        print(f"\nBenchmarking: {algo} @ ratio={ratio}")
        
        # Create model
        model = create_patched_model(MODEL_NAME, algo, ratio, DEVICE)
        
        # Benchmark
        throughput, avg_flops = benchmark_model(model, images, DEVICE)
        
        # Get predictions for consistency check
        probs, indices = get_predictions(model, images, DEVICE)
        
        result = {
            'algorithm': algo,
            'ratio': ratio,
            'throughput': throughput,
            'gflops': avg_flops / 1e9 if avg_flops > 0 else 'N/A',
            'top1_pred': indices[:, 0].tolist(),
            'top1_prob': probs[:, 0].tolist(),
        }
        results.append(result)
        
        print(f"  Throughput: {throughput:.1f} img/s")
        print(f"  GFLOPs: {result['gflops']}")
        
        # Free memory
        del model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create DataFrame
df = pd.DataFrame(results)
df = df[df['ratio'] != 1.0]  # Exclude baseline for ratio comparison

print("\n=== Results Summary ===")
print(df.to_string(index=False))

In [ ]:
# Plot throughput comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput vs Ratio
ax1 = axes[0]
for algo in ['pitome', 'tda_pitome']:
    algo_df = df[df['algorithm'] == algo]
    ax1.plot(algo_df['ratio'], algo_df['throughput'], 'o-', label=algo, linewidth=2, markersize=8)

ax1.set_xlabel('Keep Ratio', fontsize=12)
ax1.set_ylabel('Throughput (img/s)', fontsize=12)
ax1.set_title('Throughput vs Keep Ratio', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Bar chart comparison at ratio=0.9
ax2 = axes[1]
ratio_09 = df[df['ratio'] == 0.9]
if len(ratio_09) > 0:
    x = range(len(ratio_09))
    bars = ax2.bar(x, ratio_09['throughput'], color=['#3498db', '#e74c3c'])
    ax2.set_xticks(x)
    ax2.set_xticklabels(ratio_09['algorithm'], fontsize=12)
    ax2.set_ylabel('Throughput (img/s)', fontsize=12)
    ax2.set_title('Throughput at Ratio=0.9', fontsize=14)
    
    # Add value labels
    for bar, val in zip(bars, ratio_09['throughput']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Full ImageNet Evaluation (Optional)

For proper accuracy measurement, run the full evaluation script:

In [ ]:
# Full evaluation on ImageNet (requires ImageNet dataset)
# Uncomment to run:

# !python main_ic.py --eval --algo pitome --model DEIT-S-224 --ratio 0.9 --batch-size 128
# !python main_ic.py --eval --algo tda_pitome --model DEIT-S-224 --ratio 0.9 --batch-size 128

## 8. Summary

| Metric | PiToMe | TDA-PiToMe | Notes |
|--------|--------|------------|-------|
| **Accuracy** | Baseline | ~Same or better | Uses topological importance |
| **Throughput** | Faster | Slightly slower | TDA scoring adds overhead |
| **Scoring** | Energy-based | Topology-aware | Captures multi-scale structure |